# Univariate CNN model

Reference: Jason Brownlee. "Deep Learning for Time Series Forecasting: Predict the Future with MLPs, CNNs, and LSTMs in Python".

>Although traditionally developed for two-dimensional image data, CNNs can be used to model
univariate time series forecasting problems. Univariate time series are datasets comprised of a
single series of observations with a temporal ordering and a model is required to learn from the
series of past observations to predict the next value in the sequence. This section is divided into
two parts; they are Data Preparation and CNN model.
>
>Before a univariate series can be modeled, it must be prepared. The CNN model will learn a
function that maps a sequence of past observations as input to an output observation. As such,
the sequence of observations must be transformed into multiple examples from which the model
can learn. Consider a given univariate sequence:

In [1]:
# transform univariate time series to supervised learning problem
from numpy import array

# split a univariate sequence into samples
def split_sequence(sequence, n_steps):
    X, y = list(), list()
    for i in range(len(sequence)):
        # find the end of this pattern
        end_ix = i + n_steps
        # check if we are beyond the sequence
        if end_ix > len(sequence)-1:
            break
        # gather input and output parts of the pattern
        seq_x, seq_y = sequence[i:end_ix], sequence[end_ix]
        X.append(seq_x)
        y.append(seq_y)
    return array(X), array(y)

# define input sequence
raw_seq = [10, 20, 30, 40, 50, 60, 70, 80, 90]
# choose a number of time steps
n_steps = 3
# split into samples
X, y = split_sequence(raw_seq, n_steps)
# summarize the data
for i in range(len(X)):
    print(X[i], y[i])

n_features = 1
X = X.reshape((X.shape[0], X.shape[1], n_features))
print(X.shape)

[10 20 30] 40
[20 30 40] 50
[30 40 50] 60
[40 50 60] 70
[50 60 70] 80
[60 70 80] 90
(6, 3, 1)


>Running the example splits the univariate series into six samples where each sample has
three input time steps and one output time step. Now that we know how to prepare a univariate series for modeling, let’s look at developing
a CNN model that can learn the mapping of inputs to outputs.
>
>A one-dimensional CNN is a CNN model that has a convolutional hidden layer that operates
over a 1D sequence. This is followed by perhaps a second convolutional layer in some cases,
such as very long input sequences, and then a pooling layer whose job it is to distill the output
of the convolutional layer to the most salient elements. The convolutional and pooling layers are followed by a dense fully connected layer that interprets the features extracted by the
convolutional part of the model. A flatten layer is used between the convolutional layers and
the dense layer to reduce the feature maps to a single one-dimensional vector. We can define a
1D CNN Model for univariate time series forecasting as follows.
>
>Key in the definition is the shape of the input; that is what the model expects as input for
each sample in terms of the number of time steps and the number of features. We are working
with a univariate series, so the number of features is one, for one variable. The number of
time steps as input is the number we chose when preparing our dataset as an argument to the
split `sequence()` function.
>
>The input shape for each sample is specified in the input shape argument on the definition
of the first hidden layer. We almost always have multiple samples, therefore, the model will
expect the input component of training data to have the dimensions or shape: `[samples,
timesteps, features]`. Our `split sequence()` function in the previous section outputs the
$X$ with the shape `[samples, timesteps]`, so we can easily reshape it to have an additional
dimension for the one feature.
>
>The CNN does not actually view the data as having time steps, instead, it is treated as a
sequence over which convolutional read operations can be performed, like a one-dimensional
image. In this example, we define a convolutional layer with 64 filter maps and a kernel size
of 2. This is followed by a max pooling layer and a dense layer to interpret the input feature.
An output layer is specified that predicts a single numerical value. The model is fit using the
eﬃcient Adam version of stochastic gradient descent and optimized using the mean squared
error, or ‘mse’, loss function. Once the model is defined, we can fit it on the training dataset.
>
>After the model is fit, we can use it to make a prediction. We can predict the next value
in the sequence by providing the input: `[70, 80, 90]`. And expecting the model to predict
something like: `[100]`. The model expects the input shape to be three-dimensional with
`[samples, timesteps, features]`, therefore, we must reshape the single input sample before
making the prediction.
>
>We can tie all of this together and demonstrate how to develop a 1D CNN model for
univariate time series forecasting and make a single prediction.

In [2]:
# univariate cnn example
from numpy import array
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Conv1D, MaxPooling1D, Input

# Define model
model = Sequential()
model.add(Input(shape=(n_steps, n_features))) 
model.add(Conv1D(filters=64, kernel_size=2, activation='relu'))
model.add(MaxPooling1D(pool_size=2))
model.add(Flatten())
model.add(Dense(50, activation='relu'))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse')

# Fit model
print("Training model...")
model.fit(X, y, epochs=1000, verbose=0)
print("Training complete!")

# Demonstrate prediction
x_input = array([70, 80, 90])
x_input = x_input.reshape((1, n_steps, n_features))

yhat = model.predict(x_input, verbose=0)
print(f"Predicted value: {yhat}")

model.summary()

/Users/thomschu/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Training model...
Training complete!
Predicted value: [[103.13909]]


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 2, 64)          │           192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 1, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 50)             │         3,250 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,481 (40.95 KB)

 Trainable params: 3,493 (13.64 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 6,988 (27.30 KB)

**`model = Sequential()`**
* **What it does:** This initializes an empty neural network. The `Sequential` name means that we are going to build this model layer-by-layer in a straight line, where the output of one layer flows directly into the input of the next.

**`model.add(Input(shape=(n_steps, n_features)))`**
* **What it does:** This is the loading dock of your assembly line. It doesn't do any math; it simply tells the network exactly what shape the incoming data will be so the network can prepare its internal structure. As we established earlier, it expects `n_steps` (the length of your sliding window) and `n_features` (how many variables you are tracking).

**`model.add(Conv1D(filters=64, kernel_size=2, activation='relu'))`**
* **What it does:** This is the core pattern-finder. 
    * `kernel_size=2`: This creates a "window" that looks at 2 time steps at a time. It slides down your time series sequence step-by-step.
    * `filters=64`: The layer will use 64 different versions of this sliding window. Each filter acts like a specific magnifying glass trying to learn a different pattern (like an upward trend, a sudden drop, etc.).
    * `activation='relu'`: This stands for Rectified Linear Unit. It is a mathematical filter that says "if a number is negative, turn it to zero; if it's positive, leave it alone." This helps the network learn complex, non-linear patterns rather than just drawing straight lines.

**`model.add(MaxPooling1D(pool_size=2))`**
* **What it does:** This layer is a summarizer.  It takes the output from the `Conv1D` layer, looks at it in chunks of 2 (`pool_size=2`), and only keeps the maximum value from each chunk. This shrinks the data down, throwing away irrelevant noise and keeping only the strongest, most important features. It makes the model faster and helps prevent overfitting (memorizing the data).

**`model.add(Flatten())`**
* **What it does:** This is the bridge between the Convolutional layers (which output 3D data) and the Dense layers (which only accept 1D data).  It literally "flattens" your data, taking the multi-dimensional output of the MaxPooling layer and crushing it into a single, long, 1D array of numbers. 

**`model.add(Dense(50, activation='relu'))`**
* **What it does:** This is a standard, fully connected neural network layer (like the ones students usually see in introductory diagrams). It has 50 individual "neurons" (nodes) that look at the flattened array of features and learn how those features combine to influence the final outcome. It also uses the `relu` activation to process the signals.

**`model.add(Dense(1))`**
* **What it does:** This is the final output layer. Because you are trying to predict a single, continuous number (like the number of sales for the next hour), you only need 1 neuron. It takes the complex thoughts from the previous 50 neurons and condenses them into your final predicted value.

**`model.compile(optimizer='adam', loss='mse')`**
* **What it does:** The assembly line is built, but it doesn't know *how* to learn yet. This line finalizes the model.
    * `optimizer='adam'`: This is the engine that updates the model's internal weights during training. 'Adam' is currently the industry-standard algorithm because it is very efficient and adapts well to different problems.
    * `loss='mse'`: This stands for Mean Squared Error. It tells the model how to measure its mistakes. After every guess, it squares the difference between its prediction and the actual true value. The optimizer's goal is to make this MSE number as close to zero as possible!